In [1]:
from sage.all import *
import sage.libs.lrcalc.lrcalc as lrcalc


In [2]:
def degree(partition):
    return sum(partition)

def compare_by_degree(p1, p2):
    d1, d2 = degree(p1), degree(p2)
    if d1 < d2:
        return -1
    elif d1 > d2:
        return 1
    return 0


In [3]:
degree((3,))

3

In [4]:

def generate_partitions(n, max_part=None):
    """Generate all partitions of n."""
    if n == 0:
        yield ()
    else:
        if max_part is None or max_part > n:
            max_part = n
        for first in range(max_part, 0, -1):
            for rest in generate_partitions(n - first, first):
                yield (first,) + rest

p = (3, 1) 
n=5
parts = list(generate_partitions(5))
print(parts)




[(5,), (4, 1), (3, 2), (3, 1, 1), (2, 2, 1), (2, 1, 1, 1), (1, 1, 1, 1, 1)]


In [5]:
def subset_partitions(partition):
    """Return all partitions with degree <= degree(partition)."""
    d = degree(partition)
    result = []
    for n in range(d + 1):
        result.extend(generate_partitions(n))
    return result


subset_partitions((3,1))

[(),
 (1,),
 (2,),
 (1, 1),
 (3,),
 (2, 1),
 (1, 1, 1),
 (4,),
 (3, 1),
 (2, 2),
 (2, 1, 1),
 (1, 1, 1, 1)]

In [6]:

print("Degree of p:", degree(p))

q = (1, 1)
print("Compare p and q:", compare_by_degree(p, q))  # 0 means equal

subset = subset_partitions(p)
print(f"All partitions with degree ≤ {degree(p)}:")
print(subset)


Degree of p: 4
Compare p and q: 1
All partitions with degree ≤ 4:
[(), (1,), (2,), (1, 1), (3,), (2, 1), (1, 1, 1), (4,), (3, 1), (2, 2), (2, 1, 1), (1, 1, 1, 1)]


# Calculating 2 LW Coefficient Manually Using COmbinatorics

In [7]:

from itertools import permutations

def is_partition(p):
    return all(p[i] >= p[i+1] for i in range(len(p)-1))

def subtract_partitions(lam, mu):
    """Return skew shape lam/mu as list of row lengths."""
    if len(mu) > len(lam) or any(mu[i] > lam[i] for i in range(len(mu))):
        return None
    skew = [lam[i] - (mu[i] if i < len(mu) else 0) for i in range(len(lam))]
    return skew

def weight_to_list(weight):
    """Expand weight partition to list, e.g. (2,1) -> [1,1,2]."""
    res = []
    for i, m in enumerate(weight):
        res += [i+1]*m
    return res

def is_yamanouchi(word):
    """Check lattice word (Yamanouchi) condition."""
    counts = {}
    for x in word:
        counts[x] = counts.get(x, 0) + 1
        for y in range(1, x):
            if counts.get(y, 0) < counts[x]:
                return False
    return True

def lrcoefP(mu, nu, lam):
    """Compute c^{lam}_{mu,nu} using Yamanouchi condition."""
    skew = subtract_partitions(lam, mu)
    if skew is None:
        return 0
    num_boxes = sum(skew)
    if num_boxes != sum(nu):
        return 0

    # naive enumeration for small cases
    entries = weight_to_list(nu)
    c = 0
    for perm in set(permutations(entries)):
        if is_yamanouchi(perm):
            c += 1
    return c

In [8]:
# Tests LW 2
print(lrcoefP((1,), (1,), (2,)))    # Expect 1  (since s_1 * s_1 = s_2 + s_{1,1})
print(lrcoefP((1,), (1,), (1,1)))   # Expect 1
print(lrcoefP((2,), (1,), (3,)))    # Expect 1
print(lrcoefP((2,), (1,), (2,1)))   # Expect 1
print(lrcoefP((2,), (2,), (4,)))    # Expect 1

1
1
1
1
1


In [9]:
def solveOne(k, k1, k2, k3, k4):
    """
    Solve the system

        2*x2 + x1 + x3 + x4 = k                (1)
        k1 - k2 = x2 + x1 + x3                  (2)
        k3 - k4 = x2 + x1 + x4                  (3)

    together with the bounds

        x1,x2,x3,x4 < k1   and   x1,x2,x3,x4 < k3
        k2 < k1 ,   k4 < k3   (these are assumed true, but are checked)

    Parameters
    ----------
    k  : int   – the constant that appears in (1)
    k1 : int   – |λ|
    k2 : int   – |λ'| (already chosen)
    k3 : int   – |μ|
    k4 : int   – |μ'| (already chosen)

    Returns
    -------
    list of dicts, each dict containing the six numbers
    {"deg δ":x1, "deg γ":x2, "p":x3, "q":x4,
     "deg λ'":k2, "deg μ'":k4}
    """
 
    # 0.  sanity checks that are required by the statement
 
    if not (k2 < k1 and k4 < k3):

        return []

 
    # 1.  Derive x1 directly from (1) + (2) + (3)
 
    # Adding (2) and (3) and subtracting (1) eliminates x2 completely:
    #   (k1 - k2) + (k3 - k4) - k = x1
    x1 = k1 - k2 + k3 - k4 - k
    if x1 < 0:                     # x1 must be non‑negative
        return []

 
    # 2.  The remaining unknown is x2 ; once we know x2 we get x3,x4
 
    # From (2)   :  x3 = (k1 - k2) - (x1 + x2)
    # From (3)   :  x4 = (k3 - k4) - (x1 + x2)
    # Both x3 and x4 must be ≥ 0, therefore
    #   x1 + x2 ≤ k1 - k2   and   x1 + x2 ≤ k3 - k4 
    max_sum = min(k1 - k2, k3 - k4)      # the largest allowed value of (x1+x2)

    # The smallest possible sum is just x1 (when x2 = 0)
    if x1 > max_sum:                     # no room for a non‑negative x2
        return []

    solutions = []
    # x2 can range from 0 up to the value that keeps the sum ≤ max_sum
    max_x2 = max_sum - x1
    for x2 in range(max_x2 + 1):         # inclusive upper bound
        s = x1 + x2                       # s = x1 + x2

        x3 = (k1 - k2) - s
        x4 = (k3 - k4) - s

        # all variables must be non‑negative
        if x3 < 0 or x4 < 0:
            continue

        # condition (4): each of x1,x2,x3,x4 < k1 and < k3
        if not (x1 < k1 and x2 < k1 and x3 < k1 and x4 < k1):
            continue
        if not (x1 < k3 and x2 < k3 and x3 < k3 and x4 < k3):
            continue

        # everything checks out – store the solution
        solutions.append({
            "deg δ": x1,
            "deg γ": x2,
            "p":    x3,
            "q":    x4,
            "deg λ'": k2,
            "deg μ'": k4
        })

    return solutions

In [10]:
# Examples: Correct Checked by Hand

print(solveOne(1, 2, 1, 2, 1))
print(solveOne(3, 3, 2, 2, 0))

# Ex1 by Professor

print(solveOne(1, 2, 2, 1, 1))

#Ex 2 by proffesor

print(solveOne(3, 1, 0, 1, 0))


[{'deg δ': 1, 'deg γ': 0, 'p': 0, 'q': 0, "deg λ'": 1, "deg μ'": 1}]
[{'deg δ': 0, 'deg γ': 1, 'p': 0, 'q': 1, "deg λ'": 2, "deg μ'": 0}]
[]
[]


In [11]:
def solveTwo(mu1, mu2, mu3, mu4):

    v1 = mu1 + mu2
    v2 = mu3 + mu4
    

    return [v1, v2]


In [12]:
# -----------------------------------------------------------------
#  three‑fold LR convolution – unchanged logic, corrected arguments
# -----------------------------------------------------------------
def lrcoef4(mu1, mu2, mu3, mu4, lam):
    """
    Return
        Σ_{a ⊢ (mu1+mu2)} Σ_{b ⊢ (mu3+mu4)}
               c^{a}_{mu1,mu2} · c^{b}_{mu3,mu4} · c^{lam}_{a,b}

    The first four arguments are *integers* (the degrees that appear in the
    linear system).  ``lam`` must be the *partition* λ (a tuple), not its
    total degree.
    """

    d_mu1 = degree(mu1)
    d_mu2 = degree(mu2)
    d_mu3 = degree(mu3)
    d_mu4 = degree(mu4)
    d_lam = degree(lam)
    
    total1 = d_mu1 + d_mu2                # degree of the first intermediate partition
    total2 = d_mu3 + d_mu4                # degree of the second intermediate partition

    parts_a = list(generate_partitions(total1))
    parts_b = list(generate_partitions(total2))


    # ----- DEBUG -------
    print("sum mu:", mu1+mu2+mu3+mu4, "sum lam:", sum(lam))

    print(f'possible partitions for v1: {parts_a}\n')
    print(f'possible partitions for v1: {parts_b}\n')
    print(type(parts_a[0]), parts_a[0])



    s = 0
    for a in parts_a:
        c1 = int(lrcoefP(mu1, mu2, a))   # c^{a}_{mu1,mu2}
        if c1 == 0:
            continue
        for b in parts_b:
            c2 = int(lrcoefP(mu3, mu4, b))   # c^{b}_{mu3,mu4}
            if c2 == 0:
                continue
            c3 = int(lrcoefP(a, b, lam))           # c^{lam}_{a,b}
            if c3 == 0:
                continue
            s += c1 * c2 * c3
    return s


In [13]:
print(f'{lrcoef4((1,), (1,), (1,), (1,), (2,2))} done! \n')


print(lrcoefP((1,), (1,), (2,)))




sum mu: (1, 1, 1, 1) sum lam: 4
possible partitions for v1: [(2,), (1, 1)]

possible partitions for v1: [(2,), (1, 1)]

<class 'tuple'> (2,)
4 done! 

1


In [14]:
def ones_partition(x):
    """Return the partition (1, 1, ..., 1) with x ones."""
    if x < 0:
        raise ValueError("x must be nonnegative")
    return tuple(1 for _ in range(x))


In [15]:
def CalcSoc(k, lam, lamP, mu, muP):
    """
    k1 = |lam|,   k3 = |mu|
    """

    print(f'k value: {k} \n')
    k1 = degree(lam)
    k2 = degree(lamP)
    k3 = degree(mu)
    k4 = degree(muP)

    print (f'Values of deg lam, deg lamP, deg mu, deg muP, {k1, k2, k3, k4} \n')
    
    L = solveOne(k, k1, k2, k3, k4)

    print(L)

    tot_sum = 0
    for sol in L:
        d_list = list(generate_partitions(sol["deg δ"]))
        g_list = list(generate_partitions(sol["deg γ"]))
        p = ones_partition(sol["p"])
        q = ones_partition(sol["q"])

        for d in d_list:
            for g in g_list:

                print(f'solution partitions {d, g, p, q}\n')


                # first factor uses the *full* partition lam
                term1 = lrcoef4(d, g, p, lamP, lam)
                print(f'term1 in product {term1}')

                # second factor uses the *full* partition mu
                term2 = lrcoef4(d, g, q, muP, mu)
                print(f'term2 in product {term2}')

                tot_sum += term1 * term2
    return tot_sum


In [16]:
def Master1(k, lam, lamP, mu, muP, ex_string):
    print(f'\n{ex_string} \n')
    print(solveOne(k, degree(lam), degree(lamP), degree(mu), degree(muP)))
    print(CalcSoc(k, lam, lamP, mu, muP))
    print("\n --------------------Done!-------------------- !!\n")


# My Example with Master

Master1(3, (1, 2), (1, 1), (2,), (), 'My Example')
# My Example

# Ex 1

print(f'{CalcSoc(1, (2,), (2,), (1,), (1,))}')

Master1(1, (2,), (2,), (1,), (1,), 'Example 1')


# Ex 2

Master1(3, (1,), (), (1,), (), 'Example 2')


# Ex 3

Master1(2, (1,), (), (1,), (), 'Example 3')

# Ex 4

Master1(2, (1,1), (), (1,), (), 'Example 4')





My Example 

[{'deg δ': 0, 'deg γ': 1, 'p': 0, 'q': 1, "deg λ'": 2, "deg μ'": 0}]
k value: 3 

Values of deg lam, deg lamP, deg mu, deg muP, (3, 2, 2, 0) 

[{'deg δ': 0, 'deg γ': 1, 'p': 0, 'q': 1, "deg λ'": 2, "deg μ'": 0}]
solution partitions ((), (1,), (), (1,))

sum mu: (1, 1, 1) sum lam: 3
possible partitions for v1: [(1,)]

possible partitions for v1: [(2,), (1, 1)]

<class 'tuple'> (1,)
term1 in product 2
sum mu: (1, 1) sum lam: 2
possible partitions for v1: [(1,)]

possible partitions for v1: [(1,)]

<class 'tuple'> (1,)
term2 in product 1
2

 --------------------Done!-------------------- !!

k value: 1 

Values of deg lam, deg lamP, deg mu, deg muP, (2, 2, 1, 1) 

[]
0

Example 1 

[]
k value: 1 

Values of deg lam, deg lamP, deg mu, deg muP, (2, 2, 1, 1) 

[]
0

 --------------------Done!-------------------- !!


Example 2 

[]
k value: 3 

Values of deg lam, deg lamP, deg mu, deg muP, (1, 0, 1, 0) 

[]
0

 --------------------Done!-------------------- !!


Example 3 

[]
k 